In [1]:
import torch
import os
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import timm
import huggingface_hub

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt 

from timm import create_model  # or torchvision.models
from tqdm.notebook import tqdm
from torchinfo import summary

from torch.utils.data import DataLoader, random_split

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('../project/src'))
sys.path.insert(0, os.path.abspath('../project'))

import yaml
with open('../project/config/config.yaml') as f:
    cfg = yaml.safe_load(f)
print(cfg)

{'seed': 42, 'deterministic': True, 'data': {'train_meta': 'data_meta_splits/train_test_split.csv', 'val_meta': 'data_meta_splits/train_test_split.csv', 'eeg_dir': 'data_eeg_preprocessed/eeg_npy', 'spec_dir': 'data_eeg_preprocessed/spec_npy'}, 'model': {'name': 'eeg_1d_small', 'num_classes': 6}, 'loss': {'name': 'ce', 'class_weight': [], 'label_smoothing': 0.0}, 'train': {'run_name': 'exp', 'out_dir': 'runs', 'batch_size': 64, 'num_workers': 8, 'epochs': 50, 'lr': 0.0015, 'weight_decay': 0.01, 'amp': True, 'grad_clip': 1.0, 'use_weighted_sampler': False, 'early_stop_patience': 10, 'early_stop_key': 'macro_f1', 'lr_mode': 'cos'}, 'log': {'use_wandb': False, 'project': 'EEG-HBA'}}


In [3]:
# ========== ENVIRONMENT CONFIG ==========
IS_KAGGLE = os.path.exists('/kaggle')

if IS_KAGGLE:
    DATA_ROOT = '/kaggle/input/hms-harmful-brain-activity-classification'
    META_PATH = '/kaggle/input/hms-eeg-code/data_meta_splits/train_test_split.csv'
else:
    DATA_ROOT = os.path.abspath('../')
    META_PATH = os.path.abspath('../data_meta_splits/train_test_split.csv')

# ========== REPRODUCIBILITY ==========
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'} | Device: {DEVICE}")

Environment: Local | Device: cuda


In [4]:
from data import EEGDataset
from torch.utils.data import DataLoader

EEG_DIR = os.path.join(DATA_ROOT, cfg["data"]["eeg_dir"])

train_ds = EEGDataset(META_PATH, EEG_DIR, split="train")
val_ds   = EEGDataset(META_PATH, EEG_DIR, split="val")

train_loader = DataLoader(train_ds, batch_size=cfg["train"]["batch_size"],
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg["train"]["batch_size"],
                          shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

Train: 66251 | Val: 17179


In [5]:
# Batch shape verification
batch = next(iter(train_loader))
print("x shape:", batch["x"].shape)
print("y shape:", batch["y"].shape)
print("y sample:", batch["y"][:4])

x shape: torch.Size([64, 20, 10000])
y shape: torch.Size([64])
y sample: tensor([4, 2, 4, 1])


In [6]:
from models.classifier import build_model

model = build_model(cfg["model"]).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters — total: {total_params:,} | trainable: {trainable:,}")

Parameters — total: 217,030 | trainable: 217,030


In [9]:
# 重新检查fix后的状态
batch = next(iter(train_loader))
x = batch["x"]
print("NaN in batch:", torch.isnan(x).any().item())
print("Input range:", x.min().item(), "~", x.max().item())

# 不用AMP，手动跑一个forward
model.train()
x_gpu = x.to(DEVICE)
y_gpu = batch["y"].to(DEVICE)

with torch.no_grad():
    logits = model(x_gpu)
    
print("Logits NaN:", torch.isnan(logits).any().item())
print("Logits range:", logits.min().item(), "~", logits.max().item())

loss = nn.CrossEntropyLoss()(logits, y_gpu)
print("Loss:", loss.item())

NaN in batch: False
Input range: -18300.1796875 ~ 1762811.25
Logits NaN: True
Logits range: nan ~ nan
Loss: nan


In [7]:
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import f1_score
import time

NUM_EPOCHS  = cfg["train"]["epochs"]
LR          = cfg["train"]["lr"]
GRAD_CLIP   = cfg["train"]["grad_clip"]
USE_AMP     = cfg["train"]["amp"] and DEVICE.type == "cuda"

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=LR,
                  weight_decay=cfg["train"]["weight_decay"])
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
scaler    = GradScaler(enabled=USE_AMP)

best_f1, history = 0.0, []

for epoch in range(1, NUM_EPOCHS + 1):
    # --- train ---
    model.train()
    t0 = time.time()
    train_loss = 0.0
    for batch in train_loader:
        x = batch["x"].to(DEVICE)
        y = batch["y"].to(DEVICE)
        optimizer.zero_grad()
        with autocast(enabled=USE_AMP):
            logits = model(x)
            loss   = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()
    scheduler.step()

    # --- validate ---
    model.eval()
    val_loss, all_preds, all_labels = 0.0, [], []
    with torch.no_grad():
        for batch in val_loader:
            x = batch["x"].to(DEVICE)
            y = batch["y"].to(DEVICE)
            with autocast(enabled=USE_AMP):
                logits = model(x)
                loss   = criterion(logits, y)
            val_loss  += loss.item()
            all_preds .extend(logits.argmax(1).cpu().tolist())
            all_labels.extend(y.cpu().tolist())

    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    avg_train = train_loss / len(train_loader)
    avg_val   = val_loss   / len(val_loader)
    elapsed   = time.time() - t0

    history.append({"epoch": epoch, "train_loss": avg_train,
                    "val_loss": avg_val, "macro_f1": macro_f1})

    print(f"Epoch {epoch:03d} | train_loss {avg_train:.4f} | "
          f"val_loss {avg_val:.4f} | macro_f1 {macro_f1:.4f} | {elapsed:.0f}s")

    if macro_f1 > best_f1:
        best_f1 = macro_f1
        torch.save(model.state_dict(), "best_model.pt")
        print(f"  ✓ saved best model (f1={best_f1:.4f})")

print(f"\nTraining complete. Best macro_f1: {best_f1:.4f}")

C:\Users\xiaos\AppData\Local\Temp\ipykernel_28332\116416347.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = GradScaler(enabled=USE_AMP)
C:\Users\xiaos\AppData\Local\Temp\ipykernel_28332\116416347.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):
C:\Users\xiaos\AppData\Local\Temp\ipykernel_28332\116416347.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):


Epoch 001 | train_loss nan | val_loss nan | macro_f1 0.0515 | 243s
  ✓ saved best model (f1=0.0515)


C:\Users\xiaos\AppData\Local\Temp\ipykernel_28332\116416347.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):


KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

epochs     = [h["epoch"]      for h in history]
train_loss = [h["train_loss"] for h in history]
val_loss   = [h["val_loss"]   for h in history]
macro_f1   = [h["macro_f1"]   for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, train_loss, label="train")
ax1.plot(epochs, val_loss,   label="val")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.set_title("Loss Curve"); ax1.legend()

ax2.plot(epochs, macro_f1, color="green")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Macro F1")
ax2.set_title("Validation Macro F1")

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()